                # COPRO

                Propose instruction variants, evaluate them, and iteratively refine the best candidates.

                **Use it when:** The instruction is likely the bottleneck and you want a direct, interpretable prompt search without demo search.

                **What compilation changes:** Instruction text only; the selected program has no demonstrations in this benchmark.

                Important configuration in this benchmark:

                - full-profile breadth 4 and depth 2
- GPT-5.6 Sol proposes; Luna executes
- validation uses the exact-match metric

                The 74-row dataset and pair-grouped train/validation/test membership are frozen.
                The test partition is deliberately baseline-adversarial, so these scores teach
                optimizer tradeoffs; they are not a general-purpose AI-detector leaderboard.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "benchmark_summary.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import (
    artifact_paths,
    benchmark_snapshot,
    learned_program_preview,
    verify_prompt_artifact,
)

OPTIMIZER = 'copro'
print(f"optimizer={OPTIMIZER!r}")
print("reading the checked-in chapter result; no API calls")

optimizer='copro'
reading the checked-in chapter result; no API calls


                ## Compile shape

                This is the essential DSPy call used by the shared runner (setup variables omitted):

                ```python
                optimizer = dspy.COPRO(
    prompt_model=reflection_lm, metric=exact_match,
    breadth=profile.copro_breadth, depth=profile.copro_depth, track_stats=True,
)
optimized_detector = optimizer.compile(detector, trainset=trainset)
                ```

                `compile` returns a program. Calling that program on the untouched test examples is
                a separate phase; the notebook reports optimization cost/time separately from inference latency.

In [2]:
print(benchmark_snapshot(OPTIMIZER))
print()
print(artifact_paths(OPTIMIZER))

COPRO — frozen full-profile rerun
status: completed
task model: openai/gpt-5.6-luna
test accuracy: 50.0%
uplift: +0.0 points vs Luna reference
optimization: $0.5491 and 12.6min
inference latency: mean 2.31s; p95 3.05s
reload checks: prompt=True; model=None; predictions=3/3 labels

Published artifacts:
- canonical program snapshot: chapter06/optimized_programs/final/copro.json
- canonical prompt: chapter06/results/final_prompts/copro.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Read the learned instruction before the score. COPRO's artifact is especially auditable because its gain must come from wording rather than hidden examples.

The next cell shows a bounded readable preview. The complete, lossless prompt and
saved program snapshot remain at the paths printed above.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (509 characters):
Analyze the supplied passage for signals associated with AI-generated versus human-written text, including repetitiveness, formulaic structure, generic phrasing, unnatural consistency, unsupported specificity, stylistic variation, and coherent personal context. Weigh the evidence on both sides and make the best binary classification, while recognizing that authorship cannot be determined with certainty from prose alone. Return only `AI-generated` or `Human-written`; do not add explanations or qualifiers.

Demonstrations: 0

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_state_equal': True, 'mismatches': []}


## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Evaluate the returned program on a test set that was not
used during compilation, and compare accuracy, compile cost, and inference latency
rather than treating a single score as the whole result.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`. Raw provider
transcripts and temporary training outputs are intentionally excluded from the
student download.